# 01 · Розвідувальний аналіз даних (EDA)

**Задача:** передбачити швидкість адопції тварини (`AdoptionSpeed`) на PetFinder.my
за текстовим описом та фотографіями. Метрика — *quadratic weighted kappa* (QWK).

**Мета цього ноутбука:** зрозуміти структуру даних, розподіл цільової змінної,
особливості текстів і зображень, щоб обґрунтовано спланувати моделювання.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *
from src.utils import seed_everything, qwk, OptimizedRounder, write_submission
from src.data import load_data, build_image_index, add_photo_count, add_folds, text_meta_features

seed_everything(SEED)
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

## 1. Завантаження даних

In [ ]:
train, test = load_data()
print("train:", train.shape, " test:", test.shape)
print("test має таргет:", TARGET_COL in test.columns)
train.head(3)

## 2. Цільова змінна

Перевіряємо діапазон класів (0–4 чи 1–4) та дисбаланс.

In [ ]:
vc = train[TARGET_COL].value_counts().sort_index()
display(vc.to_frame("count").assign(share=lambda d: (d["count"] / len(train)).round(3)))
vc.plot.bar(title="Розподіл AdoptionSpeed");

## 3. Текстові описи

Довжина, порожні описи, мова, приклади по класах.

In [ ]:
meta = text_meta_features(train[TEXT_COL])
train_m = pd.concat([train, meta], axis=1)
display(meta.describe().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train_m["desc_words"].clip(upper=300), bins=50, ax=axes[0]).set_title("Кількість слів")
sns.boxplot(data=train_m, x=TARGET_COL, y="desc_words", ax=axes[1]).set(ylim=(0, 300), title="Слова vs клас")
plt.tight_layout();

In [ ]:
# Empty descriptions and non-ASCII share by target class
train_m.groupby(TARGET_COL)[["desc_is_empty", "desc_non_ascii", "desc_words"]].mean().round(3)

In [ ]:
# A few random examples per class to get a feel for the texts
for c in sorted(train[TARGET_COL].unique()):
    print(f"=== AdoptionSpeed = {c} ===")
    for t in train.loc[train[TARGET_COL] == c, TEXT_COL].sample(2, random_state=SEED):
        print(" -", t[:250].replace("\n", " "))
    print()

## 4. Зображення

Скільки фото на тварину, скільки тварин без фото, розміри зображень.

In [ ]:
img_index = build_image_index()
print("images:", len(img_index), " pets with photos:", img_index[ID_COL].nunique())
img_index.head()

In [ ]:
train_m = add_photo_count(train_m, img_index)
test_p = add_photo_count(test, img_index)

print("train без фото:", (train_m["n_photos"] == 0).mean().round(3))
print("test  без фото:", (test_p["n_photos"] == 0).mean().round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=train_m["n_photos"].clip(upper=10), ax=axes[0]).set_title("Фото на тварину")
train_m.groupby("n_photos")[TARGET_COL].mean().loc[:10].plot(ax=axes[1], marker="o", title="Середній клас vs n_photos")
plt.tight_layout();

In [ ]:
# Image sizes on a random sample
from PIL import Image

sizes = [Image.open(p).size for p in img_index["path"].sample(200, random_state=SEED)]
pd.DataFrame(sizes, columns=["w", "h"]).describe().T

In [ ]:
# Show several photos with their descriptions
sample = train_m[train_m["n_photos"] > 0].sample(6, random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    path = img_index.loc[img_index[ID_COL] == row[ID_COL], "path"].iloc[0]
    ax.imshow(Image.open(path)); ax.axis("off")
    ax.set_title(f"speed={row[TARGET_COL]}  n_photos={row['n_photos']}\n{row[TEXT_COL][:60]}...", fontsize=9)
plt.tight_layout();

## 5. Збереження фолдів та мета-ознак

Фолди фіксуємо один раз, щоб усі наступні ноутбуки використовували однакову валідацію.

In [ ]:
train_f = add_folds(train_m)
train_f.to_parquet(PROCESSED_DIR / "train_meta.parquet", index=False)
test_p.to_parquet(PROCESSED_DIR / "test_meta.parquet", index=False)
img_index.to_parquet(PROCESSED_DIR / "image_index.parquet", index=False)
train_f["fold"].value_counts().sort_index()

## Висновки

_(заповнити після аналізу)_

- Розподіл таргету: …
- Тексти: …
- Зображення: …
- Що це означає для моделювання: …